# 21. Финальный Real-only контроль

Этот ноутбук сохраняет проверенную реализацию исходного эксперимента и имена его
артефактов. Запускайте **Restart Kernel and Run All Cells** после выполнения всех
предыдущих пронумерованных ноутбуков.

Все входы, кроме исходных raw-данных из `config/raw_sources.json`, создаются внутри
этого проекта. Результаты записываются в `outputs/`, а модели — в `checkpoints/`.

In [1]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src").exists():
    raise RuntimeError("Неверный путь.")

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

from project_paths import *
ensure_project_directories()

print("Project root:", PROJECT_ROOT)

Project root: D:\Users\user\Desktop\DS_XRD_project


# Real-only v2 с новыми SG-метками RRUFF
Обучение той же архитектуры XRDNetV2 с нуля только на реальных размеченных данных после добавления SG-меток RRUFF. Файлы старого real-only эксперимента не перезаписываются.

Запустите `MODE = 'quick'` для проверки или `MODE = 'cv'` для полного 5-fold GroupKFold. Результаты сохраняются с префиксом `real_only_v2_with_rruff_sg`.

### Данные и промежуточные вычисления

In [2]:
# Real-only v2: та же XRDNetV2, обучение с нуля только на реальных данных
import json, math, random, time
from pathlib import Path
import numpy as np, pandas as pd, torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import GroupKFold

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE, '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

# MODE='quick' запускает один фолд для проверки; MODE='cv' — полный эксперимент.
# Бюджет совпадает с исходным циклом: 15 эпох synthetic pretrain + 50 эпох real FT = 65 эпох.
MODE = 'cv'
RUN_SUFFIX = '_quick' if MODE == 'quick' else ''
OUTPUT_PREFIX = f'real_only_v2_with_rruff_sg{RUN_SUFFIX}'
BASE = PROJECT_ROOT
DATA = BASE/'data'/'preprocessed'; OUT = BASE/'outputs'; CKPT_DIR = BASE/'checkpoints'
GRID_N = 4096; W = 3; IMPUTE_LAMBDA = 1.5406
SYSTEMS = ['triclinic','monoclinic','orthorhombic','tetragonal','trigonal','hexagonal','cubic']
HP = dict(batch=128, lr=1e-4, wd=1e-4, clip=1.0, epochs=65, warmup_frac=.1, n_folds=5)

stats = json.loads((OUT/'pretrain_stats.json').read_text())
VOCAB = stats['vocab']; EL_IDX = {e:i for i,e in enumerate(VOCAB)}
LAT_MEAN = np.array(stats['lat_mean']); LAT_STD = np.array(stats['lat_std'])
VOL_MEAN, VOL_STD = stats['vol_mean'], stats['vol_std']
index = pd.read_parquet(DATA/'index_preprocessed.parquet'); N_TOTAL = len(index)
X_MM = np.memmap(DATA/'X_intensity.f16', dtype=np.float16, mode='r', shape=(N_TOTAL,GRID_N))
M_MM = np.memmap(DATA/'M_mask.u8', dtype=np.uint8, mode='r', shape=(N_TOTAL,GRID_N))
row_of = dict(zip(index.sample_id, index.row_idx))
ft = pd.read_parquet(BASE/'data'/'clean'/'ft_pool_combined_with_rruff_sg.parquet').copy()
ft['row_idx'] = ft.sample_id.map(row_of); assert ft.row_idx.notna().all()
ft['primary_wavelength'] = ft.primary_wavelength.fillna(IMPUTE_LAMBDA)
abc = ft[['lattice_a','lattice_b','lattice_c']].to_numpy(float)
ang = ft[['alpha','beta','gamma']].to_numpy(float)
ca,cb,cg = (np.cos(np.radians(ang[:,i])) for i in range(3))
t = 1-ca**2-cb**2-cg**2+2*ca*cb*cg
ft['V'] = abc.prod(1)*np.sqrt(np.clip(t,1e-12,None)); ft['lat6'] = list(np.hstack([np.log(abc),ang]))

device: cuda | NVIDIA GeForce RTX 3060 Ti


### Функция `to_list`

In [3]:
def to_list(v):
    if isinstance(v,str):
        try: return json.loads(v)
        except Exception: return []
    if isinstance(v,(list,tuple,np.ndarray)): return [e for e in v if isinstance(e,str)]
    return []

### Данные и промежуточные вычисления

In [4]:
ft['elements'] = ft.elements_list.apply(to_list)
ft['conn_key'] = ft.phase_compositions.astype(str)+'|'+ft.lattice_a.round(2).astype(str)
rruff_mask = ft['dataset_role'].astype(str).str.lower().eq('rruff')
assert 'rruff_mineral_name' in ft.columns, (
    'Нет rruff_mineral_name: сначала обновите RRUFF parquet новыми SG-метками')
assert ft.loc[rruff_mask, 'rruff_mineral_name'].notna().all(), 'Есть RRUFF-строки без названия минерала'
ft.loc[rruff_mask, 'conn_key'] = ('rruff_mineral|' +
    ft.loc[rruff_mask, 'rruff_mineral_name'].astype(str).str.casefold())
assert ft.loc[rruff_mask, 'spacegroup_number'].notna().sum() >= 1000, (
    'Новые SG не найдены в ft_pool_combined.parquet: сначала пересоберите parquet')
print('REAL-ONLY pool:',len(ft),'rows | groups:',ft.conn_key.nunique())

REAL-ONLY pool: 3475 rows | groups: 1584


### Функция `build_labels`

In [5]:
def build_labels(frame):
    l2 = frame.secondary_wavelength.to_numpy(float)
    lam = np.stack([frame.primary_wavelength.to_numpy(float)/1.54,
                    np.nan_to_num(l2)/1.54,np.isfinite(l2).astype(np.float32)],1).astype(np.float32)
    lat6 = np.stack(frame.lat6.to_numpy()); latm = (~np.isnan(lat6).any(1)).astype(np.float32)
    lat6 = ((lat6-LAT_MEAN)/LAT_STD).astype(np.float32)
    vol = ((np.log(frame.V.to_numpy(float))-VOL_MEAN)/VOL_STD).astype(np.float32)
    sgraw = frame.spacegroup_number.to_numpy(float); sgm = np.isfinite(sgraw).astype(np.float32)
    sg = np.nan_to_num(sgraw).astype(np.int64)-1
    smap = {s:i for i,s in enumerate(SYSTEMS)}; sr = frame.crystal_system.map(smap)
    sysm = sr.notna().to_numpy(np.float32); sys_ = sr.fillna(0).to_numpy(np.int64)
    el = np.zeros((len(frame),len(VOCAB)),np.float32); elm=np.zeros(len(frame),np.float32)
    for i,els in enumerate(frame.elements):
        if len(els):
            elm[i]=1
            for e in els:
                if e in EL_IDX: el[i,EL_IDX[e]]=1
    return dict(lam=lam,lat6=lat6,latm=latm,vol=vol,volm=latm.copy(),sg=sg,sgm=sgm,
                sys_=sys_,sysm=sysm,el=el,elm=elm)

### Класс `SpecDS`

In [6]:
class SpecDS(Dataset):
    def __init__(self, frame): self.row=frame.row_idx.to_numpy(np.int64); self.L=build_labels(frame)
    def __len__(self): return len(self.row)
    def __getitem__(self,i):
        x=np.empty((2,GRID_N),np.float32); x[0]=X_MM[self.row[i]]; x[1]=M_MM[self.row[i]]; L=self.L
        return (torch.from_numpy(x),torch.from_numpy(L['lam'][i]),torch.from_numpy(L['lat6'][i]),
                torch.tensor(L['latm'][i]),torch.tensor(L['sg'][i]),torch.tensor(L['sgm'][i]),
                torch.tensor(L['sys_'][i]),torch.tensor(L['sysm'][i]),torch.from_numpy(L['el'][i]),
                torch.tensor(L['elm'][i]),torch.tensor(L['vol'][i]),torch.tensor(L['volm'][i]))

### Класс `ResBlock`

In [7]:
class ResBlock(nn.Module):
    def __init__(self,cin,cout,stride=1):
        super().__init__(); self.conv1=nn.Conv1d(cin,cout,3,stride,1,bias=False); self.n1=nn.GroupNorm(8,cout)
        self.conv2=nn.Conv1d(cout,cout,3,padding=1,bias=False); self.n2=nn.GroupNorm(8,cout)
        self.skip=nn.Identity() if cin==cout and stride==1 else nn.Sequential(nn.Conv1d(cin,cout,1,stride,bias=False),nn.GroupNorm(8,cout))
    def forward(self,x):
        h=F.gelu(self.n1(self.conv1(x))); h=self.n2(self.conv2(h)); return F.gelu(h+self.skip(x))

### Класс `XRDNetV2`

In [8]:
class XRDNetV2(nn.Module):
    def __init__(self,n_el,w=3):
        super().__init__(); c=[32*w,48*w,64*w,96*w,128*w,192*w,256*w]
        self.stem=nn.Sequential(nn.Conv1d(2,c[0],15,padding=7,bias=False),nn.GroupNorm(8,c[0]),nn.GELU())
        self.blocks=nn.Sequential(*[ResBlock(c[i],c[i+1],2) for i in range(6)])
        self.lam_mlp=nn.Sequential(nn.Linear(3,16*w),nn.GELU(),nn.Linear(16*w,16*w))
        self.trunk=nn.Sequential(nn.Linear(c[-1]+16*w,512*w),nn.GELU(),nn.Linear(512*w,512*w),nn.GELU())
        self.head_lat=nn.Linear(512*w,6); self.head_vol=nn.Linear(512*w,1); self.head_sg=nn.Linear(512*w,230)
        self.head_sys=nn.Linear(512*w,7); self.head_el=nn.Linear(512*w,n_el)
    def forward(self,x,lam):
        f=self.blocks(self.stem(x)); w=F.adaptive_avg_pool1d(x[:,1:2],f.shape[-1]).clamp_min(1e-3)
        z=torch.cat([(f*w).sum(-1)/w.sum(-1),self.lam_mlp(lam)],1); z=self.trunk(z)
        return dict(lat=self.head_lat(z),vol=self.head_vol(z).squeeze(-1),sg=self.head_sg(z),sys=self.head_sys(z),el=self.head_el(z))

### Функция `masked_l1`

In [9]:
def masked_l1(p,t,m): return F.smooth_l1_loss(p[m>0],t[m>0]) if (m>0).sum() else p.new_zeros(())

### Функция `masked_ce`

In [10]:
def masked_ce(p,t,m): return F.cross_entropy(p[m>0],t[m>0].long()) if (m>0).sum() else p.new_zeros(())

### Функция `losses`

In [11]:
def losses(out,b):
    _,_,lat,latm,sg,sgm,sys_,sysm,el,elm,vol,volm=b; m=latm>0
    ll=F.smooth_l1_loss(out['lat'][m,:3],lat[m,:3]) if m.sum() else out['lat'].new_zeros(())
    la=F.smooth_l1_loss(out['lat'][m,3:],lat[m,3:]) if m.sum() else out['lat'].new_zeros(())
    me=elm>0; le=F.binary_cross_entropy_with_logits(out['el'][me],el[me]) if me.sum() else out['el'].new_zeros(())
    d=dict(lat=ll,ang=la,vol=masked_l1(out['vol'],vol,volm),sg=masked_ce(out['sg'],sg,sgm),sys=masked_ce(out['sys'],sys_,sysm),el=le)
    return d, ll+2*la+.5*d['vol']+d['sg']+d['sys']+d['el']

### Функция `evaluate`

In [12]:
@torch.no_grad()
def evaluate(model,loader):
    model.eval(); rows=[]
    for b in loader:
        b=[x.to(DEVICE) for x in b]
        with torch.autocast('cuda',dtype=torch.float16,enabled=DEVICE.type=='cuda'): out=model(b[0],b[1])
        _,_,lat,latm,sg,sgm,sys_,sysm,el,elm,vol,volm=b
        lp=out['lat'].float().cpu().numpy()*LAT_STD+LAT_MEAN; lp[:,:3]=np.exp(lp[:,:3])
        lt=lat.cpu().numpy()*LAT_STD+LAT_MEAN; lt[:,:3]=np.exp(lt[:,:3])
        for i in range(len(lat)):
            pe=frozenset(np.where(torch.sigmoid(out['el'][i]).cpu().numpy()>.5)[0]); te=frozenset(np.where(el[i].cpu().numpy()>.5)[0])
            rows.append(dict(sg_ok=float(sgm[i]>0 and out['sg'][i].argmax()==sg[i]),sg_top5=float(sgm[i]>0 and sg[i] in out['sg'][i].topk(5).indices),
                sys_ok=float(sysm[i]>0 and out['sys'][i].argmax()==sys_[i]),mae_a=abs(lp[i,0]-lt[i,0]) if latm[i]>0 else np.nan,
                mae_a_med=abs(lp[i,0]-lt[i,0]) if latm[i]>0 else np.nan,mae_ang=np.abs(lp[i,3:]-lt[i,3:]).mean() if latm[i]>0 else np.nan,
                pred_el=pe,true_el=te,elm=float(elm[i]),sgm=float(sgm[i]),sysm=float(sysm[i]),latm=float(latm[i])))
    model.train(); d=pd.DataFrame(rows); r={'n':len(d),'sg_n':int(d.sgm.sum())}
    for key,mask,col in [('sg_acc',d.sgm>0,'sg_ok'),('sg_top5',d.sgm>0,'sg_top5'),('sys_acc',d.sysm>0,'sys_ok')]: r[key]=d.loc[mask,col].mean() if mask.any() else np.nan
    v=d[d.latm>0]; r['mae_a']=v.mae_a.mean() if len(v) else np.nan; r['mae_a_med']=v.mae_a.median() if len(v) else np.nan; r['mae_ang']=v.mae_ang.mean() if len(v) else np.nan
    v=d[d.elm>0]; tp=sum(len(x.pred_el&x.true_el) for _,x in v.iterrows()); fp=sum(len(x.pred_el-x.true_el) for _,x in v.iterrows()); fn=sum(len(x.true_el-x.pred_el) for _,x in v.iterrows()); p=tp/max(tp+fp,1); q=tp/max(tp+fn,1); r['el_f1_micro']=2*p*q/max(p+q,1e-9); r['el_exact']=float((v.pred_el==v.true_el).mean()) if len(v) else np.nan
    return r

### Функция `train_fold`

In [13]:
def train_fold(train_frame,val_frame,epochs):
    model=XRDNetV2(len(VOCAB),W).to(DEVICE); opt=torch.optim.AdamW(model.parameters(),lr=HP['lr'],weight_decay=HP['wd']); scaler=torch.amp.GradScaler('cuda',enabled=DEVICE.type=='cuda')
    tr=DataLoader(SpecDS(train_frame),batch_size=HP['batch'],shuffle=True,pin_memory=True); va=DataLoader(SpecDS(val_frame),batch_size=HP['batch'],shuffle=False,pin_memory=True)
    total=epochs*len(tr); warm=max(1,int(total*HP['warmup_frac'])); sched=torch.optim.lr_scheduler.LambdaLR(opt,lambda s:s/warm if s<warm else .5*(1+math.cos(math.pi*min((s-warm)/max(total-warm,1),1))))
    best=-1; best_ep=0; state=None
    for ep in range(epochs):
        for b in tr:
            b=[x.to(DEVICE,non_blocking=True) for x in b]
            with torch.autocast('cuda',dtype=torch.float16,enabled=DEVICE.type=='cuda'): _,total_loss=losses(model(b[0],b[1]),b)
            opt.zero_grad(set_to_none=True); scaler.scale(total_loss).backward(); scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(),HP['clip']); scaler.step(opt); scaler.update(); sched.step()
        m=evaluate(model,va); score=(m.get('sys_acc') or 0)+(m.get('el_f1_micro') or 0)+.5*(m.get('sg_acc') or 0)
        print(f'epoch {ep+1:3d} | sys {m.get("sys_acc",np.nan):.3f} sg {m.get("sg_acc",np.nan):.3f} el {m.get("el_f1_micro",np.nan):.3f} mae_a {m.get("mae_a",np.nan):.2f}',flush=True)
        if score>best: best=score; best_ep=ep+1; state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
    model.load_state_dict(state); return model,best_ep

### Запуск эксперимента и сохранение результатов

In [14]:
folds=list(GroupKFold(n_splits=HP['n_folds']).split(ft,groups=ft.conn_key)); folds=folds[:1] if MODE=='quick' else folds
fold_metrics=[]; best_epochs=[]; t0=time.time()
for k,(tr_i,va_i) in enumerate(folds):
    print(f'\n===== REAL-ONLY FOLD {k+1}/{len(folds)} ====='); model,ep=train_fold(ft.iloc[tr_i].reset_index(drop=True),ft.iloc[va_i].reset_index(drop=True),HP['epochs']); best_epochs.append(ep)
    m=evaluate(model,DataLoader(SpecDS(ft.iloc[va_i].reset_index(drop=True)),batch_size=HP['batch'],pin_memory=True)); fold_metrics.append(m); print('best epoch',ep,'|',m); del model; torch.cuda.empty_cache()
rows=[]
for key in ['sys_acc','sg_acc','sg_top5','el_f1_micro','el_exact','mae_a','mae_a_med','mae_ang']:
    v=[m.get(key,np.nan) for m in fold_metrics]; rows.append(dict(metric=key,real_only=np.nanmean(v),std=np.nanstd(v),folds=len(v)))
summary=pd.DataFrame(rows); print(summary.to_string(index=False)); summary.to_csv(OUT/f'{OUTPUT_PREFIX}_cv_summary.csv',index=False)
old=OUT/f'ft_combined_no_replay_control_with_rruff_sg{RUN_SUFFIX}_cv_summary.csv'
if old.exists():
    prev=pd.read_csv(old); prev=prev[prev['scope'].eq('combined')] if 'scope' in prev.columns else prev; cmp=summary[['metric','real_only']].merge(prev[['metric','after_combined_ft']],on='metric',how='outer'); cmp['synthetic_pretrain_plus_real_ft']=pd.to_numeric(cmp.after_combined_ft,errors='coerce'); cmp['difference_real_only_minus_ft']=cmp.real_only-cmp.synthetic_pretrain_plus_real_ft; print('\n===== COMPARISON ====='); print(cmp.to_string(index=False)); cmp.to_csv(OUT/f'{OUTPUT_PREFIX}_vs_combined_ft.csv',index=False)
print('elapsed min:',(time.time()-t0)/60)


===== REAL-ONLY FOLD 1/5 =====
epoch   1 | sys 0.197 sg 0.000 el 0.076 mae_a 4.13
epoch   2 | sys 0.269 sg 0.036 el 0.153 mae_a 4.05
epoch   3 | sys 0.294 sg 0.079 el 0.207 mae_a 3.82
epoch   4 | sys 0.290 sg 0.056 el 0.227 mae_a 3.79
epoch   5 | sys 0.275 sg 0.069 el 0.227 mae_a 3.84
epoch   6 | sys 0.294 sg 0.082 el 0.229 mae_a 3.88
epoch   7 | sys 0.287 sg 0.066 el 0.282 mae_a 3.85
epoch   8 | sys 0.251 sg 0.036 el 0.352 mae_a 3.91
epoch   9 | sys 0.265 sg 0.072 el 0.381 mae_a 3.76
epoch  10 | sys 0.238 sg 0.066 el 0.380 mae_a 3.96
epoch  11 | sys 0.292 sg 0.056 el 0.397 mae_a 3.77
epoch  12 | sys 0.287 sg 0.118 el 0.410 mae_a 3.85
epoch  13 | sys 0.273 sg 0.102 el 0.395 mae_a 3.74
epoch  14 | sys 0.267 sg 0.082 el 0.393 mae_a 3.71
epoch  15 | sys 0.281 sg 0.056 el 0.414 mae_a 3.71
epoch  16 | sys 0.320 sg 0.100 el 0.367 mae_a 3.74
epoch  17 | sys 0.281 sg 0.100 el 0.356 mae_a 3.70
epoch  18 | sys 0.296 sg 0.107 el 0.433 mae_a 3.76
epoch  19 | sys 0.306 sg 0.084 el 0.416 mae_a 3.66